# 生存分析模型比较实验
## 36组数据 × 4种方法

In [1]:
import numpy as np
import pandas as pd
np.random.seed(2026)

import warnings
warnings.filterwarnings('ignore')
import importlib
import config
importlib.reload(config)
from config import *
import matplotlib.pyplot as plt

plt.rcParams['font.family'] = ['Heiti TC']
plt.rcParams['axes.unicode_minus'] = False

In [2]:
# 生成36组数据
random_seed = 2026
sample_sizes = [100, 500, 2000]
cens_lambdas = [0.1, 0.3, 0.5]
p_list = [1, 50]
hetero_list = [False, True]

datasets = []
configs = []

for n in sample_sizes:
    for cens in cens_lambdas:
        for p in p_list:
            for hetero in hetero_list:
                X, surv, time, event, real_censor = generate_weibull_data(
                    n=n, p=p, hetero=hetero, cens_rate=cens
                )
                datasets.append((X, surv, time, event))
                configs.append({
                    'n': n, 'cens_rate': cens, 'p': p, 
                    'hetero': hetero, 'real_censor': real_censor
                })

print(f'✅ 生成完成：{len(datasets)}组数据')

✅ 生成完成：36组数据


In [3]:
# 批量拟合四种模型
results = []

for i, (X, surv, time, event) in enumerate(datasets):
    cfg = configs[i]
    print(f"第 {i+1} 组: n={cfg['n']}, p={cfg['p']}, cens={cfg['cens_rate']}, hetero={cfg['hetero']}")

    # 1. Kaplan-Meier
    kmf = fit_kaplan_meier(time, event)
    
    # 2. Cox
    cox = fit_cox(X, time, event)
    c_cox = evaluate_model(cox, X, time, event, 'cox')
    
    # 3. Weibull
    weibull = fit_weibull(X, time, event)
    c_weibull = evaluate_model(weibull, X, time, event, 'weibull')
    
    # 4. 随机生存森林
    rsf = fit_rsf(X, time, event)
    c_rsf = evaluate_model(rsf, X, time, event, 'rsf')
    
    results.append({
        **cfg,
        'C_Cox': c_cox,
        'C_Weibull': c_weibull,
        'C_RSF': c_rsf
    })
print("数据处理完成")
df_results = pd.DataFrame(results)
df_results

第 1 组: n=100, p=1, cens=0.1, hetero=False
第 2 组: n=100, p=1, cens=0.1, hetero=True
第 3 组: n=100, p=50, cens=0.1, hetero=False
第 4 组: n=100, p=50, cens=0.1, hetero=True
第 5 组: n=100, p=1, cens=0.3, hetero=False
第 6 组: n=100, p=1, cens=0.3, hetero=True
第 7 组: n=100, p=50, cens=0.3, hetero=False
第 8 组: n=100, p=50, cens=0.3, hetero=True
第 9 组: n=100, p=1, cens=0.5, hetero=False
第 10 组: n=100, p=1, cens=0.5, hetero=True
第 11 组: n=100, p=50, cens=0.5, hetero=False
第 12 组: n=100, p=50, cens=0.5, hetero=True
第 13 组: n=500, p=1, cens=0.1, hetero=False
第 14 组: n=500, p=1, cens=0.1, hetero=True
第 15 组: n=500, p=50, cens=0.1, hetero=False
第 16 组: n=500, p=50, cens=0.1, hetero=True
第 17 组: n=500, p=1, cens=0.3, hetero=False
第 18 组: n=500, p=1, cens=0.3, hetero=True
第 19 组: n=500, p=50, cens=0.3, hetero=False
第 20 组: n=500, p=50, cens=0.3, hetero=True
第 21 组: n=500, p=1, cens=0.5, hetero=False
第 22 组: n=500, p=1, cens=0.5, hetero=True
第 23 组: n=500, p=50, cens=0.5, hetero=False
第 24 组: n=500, p=50,

,n,cens_rate,p,hetero,real_censor,C_Cox,C_Weibull,C_RSF
0,100,0.1,1,False,0.1000,0.454302,0.545698,0.741614
1,100,0.1,1,True,0.1500,0.490002,0.509998,0.764749
2,100,0.1,50,False,0.1000,0.107212,0.880225,0.948235
3,100,0.1,50,True,0.1400,0.124402,0.843472,0.956482
4,100,0.3,1,False,0.3600,0.432674,0.567326,0.752331
5,100,0.3,1,True,0.3000,0.423130,0.576870,0.727839
6,100,0.3,50,False,0.2800,0.100481,0.884286,0.956173
7,100,0.3,50,True,0.3500,0.078506,0.907959,0.955604
8,100,0.5,1,False,0.4800,0.533597,0.533597,0.809536
9,100,0.5,1,True,0.3400,0.517918,0.482082,0.731988


In [4]:
# 保存结果
df_results.to_csv('模型拟合结果.csv', index=False)
print('✅ 结果已保存')

✅ 结果已保存


In [ ]:
# CSA 实验：传统CSA方法 - 所有样本生成单侧区间[L, ∞)
# 基于加权conformal推断，提供统一覆盖率保证：P(T≥L)≥1-α

csa_results = []

# 基础模型选择：KM、Cox、Weibull、RSF
base_models = ['km', 'cox', 'weibull', 'rsf']
alpha = 0.1

for i, (X, surv, time, event) in enumerate(datasets):
    cfg = configs[i]

    print(f"CSA 第 {i+1} 组: n={cfg['n']}, p={cfg['p']}, cens={cfg['cens_rate']}, hetero={cfg['hetero']}")
    X_train, time_train, event_train, X_cal, time_cal, event_cal, X_test, time_test, event_test = split_survival_data(
        X, time, event, test_size=0.2, cal_size=0.25, random_state=2026
    )

    kmf = fit_kaplan_meier(time_train, event_train)
    cox = fit_cox(X_train, time_train, event_train)
    weibull = fit_weibull(X_train, time_train, event_train)
    rsf = fit_rsf(X_train, time_train, event_train)

    model_map = {
        'km': kmf,
        'cox': cox,
        'weibull': weibull,
        'rsf': rsf
    }

    for model_type in base_models:
        model = model_map[model_type]
        
        # 使用传统CSA方法：所有样本都是单侧区间[L, ∞)
        lower, upper, q_value = fit_csa_intervals_traditional(
            model, X_cal, time_cal, event_cal, X_test,
            alpha=alpha, model_type=model_type
        )
        
        # 使用传统CSA的覆盖率评估函数
        metrics = evaluate_interval_coverage_traditional(lower, upper, time_test, event_test)

        csa_results.append({
            'n': cfg['n'],
            'p': cfg['p'],
            'cens_rate': cfg['cens_rate'],
            'hetero': cfg['hetero'],
            'model_type': model_type,
            'method': 'Traditional CSA',
            'alpha': alpha,
            'coverage': metrics['coverage'],
            'coverage_uncensored': metrics['coverage_uncensored'],
            'coverage_censored': metrics['coverage_censored'],
            'q_value': q_value,
            'num_uncensored': metrics['num_uncensored'],
            'num_censored': metrics['num_censored']
        })


# 保存传统CSA实验结果
df_csa = pd.DataFrame(csa_results)
df_csa.to_csv('Traditional_CSA_results.csv', index=False)
print('传统CSA实验完成：', df_csa.shape[0], '条记录')

# 简要汇总
print('\n传统CSA汇总：按模型类型查看平均指标')
print(df_csa.groupby('model_type')[['coverage','coverage_uncensored','coverage_censored']].mean())
print('\n样本分布情况：')
print(df_csa.groupby('model_type')[['num_uncensored','num_censored']].mean())

CSA 第 1 组: n=100, p=1, cens=0.1, hetero=False
CSA 第 2 组: n=100, p=1, cens=0.1, hetero=True
CSA 第 3 组: n=100, p=50, cens=0.1, hetero=False
CSA 第 4 组: n=100, p=50, cens=0.1, hetero=True
CSA 第 5 组: n=100, p=1, cens=0.3, hetero=False
CSA 第 6 组: n=100, p=1, cens=0.3, hetero=True
CSA 第 7 组: n=100, p=50, cens=0.3, hetero=False
CSA 第 8 组: n=100, p=50, cens=0.3, hetero=True
CSA 第 9 组: n=100, p=1, cens=0.5, hetero=False
CSA 第 10 组: n=100, p=1, cens=0.5, hetero=True
CSA 第 11 组: n=100, p=50, cens=0.5, hetero=False
CSA 第 12 组: n=100, p=50, cens=0.5, hetero=True
CSA 第 13 组: n=500, p=1, cens=0.1, hetero=False
CSA 第 14 组: n=500, p=1, cens=0.1, hetero=True
CSA 第 15 组: n=500, p=50, cens=0.1, hetero=False
CSA 第 16 组: n=500, p=50, cens=0.1, hetero=True
CSA 第 17 组: n=500, p=1, cens=0.3, hetero=False
CSA 第 18 组: n=500, p=1, cens=0.3, hetero=True
CSA 第 19 组: n=500, p=50, cens=0.3, hetero=False
CSA 第 20 组: n=500, p=50, cens=0.3, hetero=True
CSA 第 21 组: n=500, p=1, cens=0.5, hetero=False
CSA 第 22 组: n=500, p=1